In [ ]:
%load_ext autoreload
%autoreload 2
import os

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

### Waveform distance schematics — RMSE, nRMSE, Cos Sim
Each shows two cluster-average waveforms and the shaded region representing the metric.

In [ ]:
LOW_COLOR  = '#0072B2'  # blue  — low cluster
HIGH_COLOR = '#CC79A7'  # pink  — high cluster
FILL_COLOR = '#E69F00'  # orange — fill between


def _synthetic_eap(t, peak_amp, peak_width=0.35, trough_frac=0.28, trough_delay=0.68):
    pre    = -0.13 * peak_amp * np.exp(-((t + 0.38)**2) / (2 * 0.13**2))
    peak   =  peak_amp        * np.exp(-((t       )**2) / (2 * peak_width**2))
    trough = -trough_frac * peak_amp * np.exp(-((t - trough_delay)**2) / (2 * 0.28**2))
    return pre + peak + trough


def _base_waveforms(sigma=2):
    t = np.linspace(-1.0, 1.6, 600)
    wf_high = gaussian_filter1d(_synthetic_eap(t, peak_amp=1.00, peak_width=0.30,
                                               trough_frac=0.28, trough_delay=0.66), sigma=sigma)
    wf_low  = gaussian_filter1d(_synthetic_eap(t, peak_amp=0.55, peak_width=0.44,
                                               trough_frac=0.20, trough_delay=0.72), sigma=sigma)
    return t, wf_high, wf_low


def _schematic_axes(figsize=(4, 4), lw=6, label_fontsize=22):
    fig, ax = plt.subplots(figsize=figsize)
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_color('black')
        spine.set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')
    return fig, ax, lw, label_fontsize


def plot_rmse_schematic(figsize=(4, 4), lw=6, label_fontsize=22):
    """RMSE: raw amplitude difference — waveforms at original (unscaled) amplitudes."""
    t, wf_high, wf_low = _base_waveforms()
    fig, ax, lw, label_fontsize = _schematic_axes(figsize, lw, label_fontsize)
    ax.fill_between(t, wf_low, wf_high, color=FILL_COLOR, alpha=0.30)
    ax.plot(t, wf_high, color=HIGH_COLOR, lw=lw, solid_capstyle='round')
    ax.plot(t, wf_low,  color=LOW_COLOR,  lw=lw, solid_capstyle='round')
    ax.set_xlabel('RMSE', fontsize=label_fontsize, fontweight='bold', labelpad=10,
                  fontfamily='Helvetica Neue')
    plt.tight_layout()
    return fig


def plot_nrmse_schematic(figsize=(4, 4), lw=6, label_fontsize=22):
    """nRMSE: normalized by shared global max — relative amplitude preserved."""
    t, wf_high, wf_low = _base_waveforms()
    global_max = max(np.abs(wf_high).max(), np.abs(wf_low).max())
    n_high = wf_high / global_max
    n_low  = wf_low  / global_max
    fig, ax, lw, label_fontsize = _schematic_axes(figsize, lw, label_fontsize)
    ax.fill_between(t, n_low, n_high, color=FILL_COLOR, alpha=0.30)
    ax.plot(t, n_high, color=HIGH_COLOR, lw=lw, solid_capstyle='round')
    ax.plot(t, n_low,  color=LOW_COLOR,  lw=lw, solid_capstyle='round')
    ax.set_xlabel('nRMSE', fontsize=label_fontsize, fontweight='bold', labelpad=10,
                  fontfamily='Helvetica Neue')
    plt.tight_layout()
    return fig


def plot_cossim_schematic(figsize=(4, 4), lw=6, label_fontsize=22):
    """Cos Sim: both waveforms z-scored — pure shape, amplitude removed."""
    from scipy.stats import zscore as _zscore
    t, wf_high, wf_low = _base_waveforms()
    z_high = _zscore(wf_high)
    z_low  = _zscore(wf_low)
    fig, ax, lw, label_fontsize = _schematic_axes(figsize, lw, label_fontsize)
    ax.fill_between(t, z_low, z_high, color=FILL_COLOR, alpha=0.30)
    ax.plot(t, z_high, color=HIGH_COLOR, lw=lw, solid_capstyle='round')
    ax.plot(t, z_low,  color=LOW_COLOR,  lw=lw, solid_capstyle='round')
    ax.set_xlabel('Cos Sim', fontsize=label_fontsize, fontweight='bold', labelpad=10,
                  fontfamily='Helvetica Neue')
    plt.tight_layout()
    return fig


plot_rmse_schematic()
plt.show()

plot_nrmse_schematic()
plt.show()

plot_cossim_schematic()
plt.show()